# Heavy Rainfall Prediction Using Machine Learning

## Objective

Evaluate whether recent rainfall patterns can help predict heavy rainfall
events using real IMD observations.

## Dataset

India Meteorological Department (IMD) daily gridded rainfall data.

## Study Location

Pune, Maharashtra, India.

## Target

Predict whether rainfall on the next day will be greater than or equal to
64.5 mm.

In [36]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    classification_report
)

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

In [38]:
ds = pd.read_csv("../data/imd_rainfall_2021_2025.csv")

print(ds)

            time  rainfall_mm
0     2021-01-01     0.591625
1     2021-01-02     0.020424
2     2021-01-03     0.584557
3     2021-01-04     1.516260
4     2021-01-05     1.730590
...          ...          ...
1821  2025-12-27     0.000716
1822  2025-12-28     0.000000
1823  2025-12-29     0.004724
1824  2025-12-30     0.131478
1825  2025-12-31     0.015518

[1826 rows x 2 columns]


## Extract Pune

In [44]:
PUNE_LAT = 18.52
PUNE_LON = 73.85

rainfall = ds


In [ ]:
df = rainfall.to_dataframe(
    name="rainfall_mm"
).reset_index()

df = df.rename(columns={
    "TIME": "date"
})

df["date"] = pd.to_datetime(df["date"])

# Remove invalid negative values
df["rainfall_mm"] = df["rainfall_mm"].clip(lower=0)

# Remove missing values
df = df.dropna(subset=["rainfall_mm"])

# Sort chronologically
df = df.sort_values("date").reset_index(drop=True)

df.head()

AttributeError: 'DataFrame' object has no attribute 'to_dataframe'

In [ ]:
print(df.info())
print(df.describe())
print("Total rows:", len(df))
print("Missing values:", df.isna().sum().sum())
print("Rainy days:", (df["rainfall_mm"] > 0).sum())
print("Heavy rainfall days:", (df["rainfall_mm"] >= 64.5).sum())

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    df["date"],
    df["rainfall_mm"],
    linewidth=1
)

plt.title("Daily Rainfall at the Selected Pune Grid Point")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    df["rainfall_mm"],
    bins=40
)

plt.title("Distribution of Daily Rainfall")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Number of Days")

plt.show()

In [ ]:
df["rainfall_lag_1"] = df["rainfall_mm"].shift(1)
df["rainfall_lag_2"] = df["rainfall_mm"].shift(2)
df["rainfall_lag_3"] = df["rainfall_mm"].shift(3)

df["rainfall_rolling_3d"] = (
    df["rainfall_mm"].rolling(window=3).sum()
)

df["rainfall_rolling_7d"] = (
    df["rainfall_mm"].rolling(window=7).sum()
)

df["rainfall_rolling_30d"] = (
    df["rainfall_mm"].rolling(window=30).sum()
)

In [ ]:
features = [
    "rainfall_lag_1",
    "rainfall_lag_2",
    "rainfall_lag_3",
    "rainfall_rolling_3d",
    "rainfall_rolling_7d"
]

X = df[features]
y = df["target"]

split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training positives:", y_train.sum())
print("Testing positives:", y_test.sum())

In [ ]:
HEAVY_RAINFALL_THRESHOLD = 64.5

df["target"] = (
    df["rainfall_mm"].shift(-1) >= HEAVY_RAINFALL_THRESHOLD
).astype(int)

df = df.dropna().reset_index(drop=True)

df.head()

In [ ]:
print(df["target"].value_counts())
print()
print(df["target"].value_counts(normalize=True))

positive_events = df["target"].sum()
total_events = len(df)

print("Positive events:", positive_events)
print("Total observations:", total_events)
print("Positive rate:", positive_events / total_events)

In [ ]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("Training positives:", y_train.sum())
print("Testing positives:", y_test.sum())

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=3
    )
}

In [ ]:
def evaluate_model(name, model):
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    print(f"\n{name}")
    print("-" * 40)

    print("PR-AUC:",
          average_precision_score(y_test, probabilities))

    print("ROC-AUC:",
          roc_auc_score(y_test, probabilities))

    results = []

    for threshold in [0.5, 0.3, 0.2, 0.1]:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_test,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            predictions,
            zero_division=0
        )

        results.append({
            "threshold": threshold,
            "predicted_positives": predictions.sum(),
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

    results_df = pd.DataFrame(results)

    display(results_df)

    return model, probabilities, results_df

In [ ]:
trained_models = {}
all_results = {}

for name, model in models.items():
    trained_model, probabilities, results = evaluate_model(
        name,
        model
    )

    trained_models[name] = trained_model
    all_results[name] = results

In [ ]:
comparison_rows = []

for name, results in all_results.items():
    best_row = results.loc[
        results["f1"].idxmax()
    ]

    comparison_rows.append({
        "Model": name,
        "Best Threshold": best_row["threshold"],
        "Precision": best_row["precision"],
        "Recall": best_row["recall"],
        "F1": best_row["f1"]
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results)
display(results_df.sort_values("F1", ascending=False))

In [ ]:
best_model_name = results_df.loc[
    results_df["F1"].idxmax(), "Model"
]

best_model = models[best_model_name]

print("Selected model:", best_model_name)
plt.plot(range(len(y_test)), probabilities)

## Preliminary Findings

This experiment evaluates machine learning models using real IMD rainfall
observations from the selected Pune grid point.

The primary evaluation metrics are PR-AUC, recall, precision, and F1-score,
because heavy rainfall events are relatively rare.

The results should be considered preliminary because the current dataset
contains only one year of observations. A larger multi-year dataset is
required before drawing reliable scientific conclusions.